# OpenLane-V2 causal lane-graph training
ConvNeXt V2 image encoder, Mamba-2 temporal encoder, image-only lane queries, 3D geometry, evidential lane connectivity, Hungarian supervision, and official split/evaluator integration.

Set the paths below to the official OpenLane-V2 **centerline** release and cloned devkit. CUDA and compiled Mamba dependencies are required. No experimental scores are embedded. The model predicts lane centerlines and lane-to-lane connectivity; it has no traffic-element branch, so complete OLS is not claimed.

## 1. Dependencies

In [ ]:
%pip install -q torch torchvision timm scipy numpy pandas pillow matplotlib tqdm nbformat
# Install in a CUDA toolkit environment matching PyTorch (restart kernel afterwards).
%pip install -q "mamba-ssm[causal-conv1d]" --no-build-isolation
# Clone https://github.com/OpenDriveLab/OpenLane-V2 and install its requirements.


## 2. Configuration and reproducibility

In [ ]:
import os, sys, json, random, time, hashlib, platform, subprocess
from pathlib import Path
from collections import defaultdict
import importlib.metadata as metadata
import numpy as np
import pandas as pd
from PIL import Image
from scipy.optimize import linear_sum_assignment
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from mamba_ssm import Mamba2

DATA_ROOT = Path('/path/to/OpenLane-V2')
DEVKIT_ROOT = Path('/path/to/cloned/OpenLane-V2')
SPLIT_FILE = DATA_ROOT / 'data_dict_subset_A.json'
# Use the matching trusted preprocessed devkit collection for official evaluation.
VAL_COLLECTION = DATA_ROOT / 'data_dict_subset_A_val.pkl'
RUN_DIR = Path('runs/openlanev2_causal'); RUN_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
IMAGE_SIZE = (640, 360)
SEQ_LEN = 5
MAX_LANES = 100
NUM_POINTS = 20
BATCH_SIZE = 2
EPOCHS = 100
LR = 1e-4
NUM_WORKERS = 0
EDGE_THRESHOLD = .5
LANE_THRESHOLD = .5
TEMPORAL_GATE_METERS = 2.0
# Physical bounds in ego coordinates; no target clipping is applied.
XYZ_CENTER = np.array([0., 0., 0.], dtype=np.float32)
XYZ_SCALE = np.array([50., 25., 5.], dtype=np.float32)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('Actual Mamba-2 requires a supported CUDA runtime; no substitute is used.')
DEVICE = torch.device('cuda'); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
USE_AMP = True
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
assert DATA_ROOT.is_dir() and SPLIT_FILE.is_file(), 'Set DATA_ROOT and SPLIT_FILE.'
sys.path.insert(0, str(DEVKIT_ROOT))

def git_revision(path):
    try: return subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'], text=True).strip()
    except (OSError, subprocess.CalledProcessError): return None

def write_protocol(**extra):
    versions = {}
    for package in ['torch','torchvision','timm','mamba-ssm','causal-conv1d','numpy','scipy']:
        try: versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError: versions[package] = None
    protocol = dict(seed=SEED, gpu=torch.cuda.get_device_name(), cuda=torch.version.cuda,
        python=platform.python_version(), packages=versions, image_size=IMAGE_SIZE,
        sequence_length=SEQ_LEN, max_lanes=MAX_LANES, num_points=NUM_POINTS,
        batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LR,
        split_file=str(SPLIT_FILE), split_sha256=hashlib.sha256(SPLIT_FILE.read_bytes()).hexdigest(),
        lane_threshold=LANE_THRESHOLD, edge_threshold=EDGE_THRESHOLD,
        temporal_gate_meters=TEMPORAL_GATE_METERS, xyz_center=XYZ_CENTER.tolist(),
        xyz_scale=XYZ_SCALE.tolist(), devkit_commit=git_revision(DEVKIT_ROOT),
        cudnn_deterministic=torch.backends.cudnn.deterministic,
        deterministic_algorithms=torch.are_deterministic_algorithms_enabled(), **extra)
    (RUN_DIR/'protocol.json').write_text(json.dumps(protocol, indent=2))
write_protocol(status='configured')


## 3. Official annotations, calibration, and split manifest

In [ ]:
def rigid_transform(value):
    R = np.asarray(value['rotation'], dtype=np.float64)
    t = np.asarray(value['translation'], dtype=np.float64)
    if R.shape != (3,3) or t.shape != (3,):
        raise ValueError('Expected official 3x3 rotation and 3-vector translation.')
    T = np.eye(4); T[:3,:3] = R; T[:3,3] = t
    return T

def project_ego(points, camera, image_size):
    # Official extrinsic maps camera -> ego; invert it to project ego -> camera.
    T = rigid_transform(camera['extrinsic'])
    pc = (np.asarray(points)-T[:3,3]) @ T[:3,:3]
    K = np.asarray(camera['intrinsic']['K'], dtype=np.float64)
    uvw = pc @ K.T
    uv = uvw[:,:2] / np.maximum(pc[:,2:3], 1e-8)
    w,h = image_size
    valid = (pc[:,2]>1e-5) & np.isfinite(uv).all(1)
    valid &= (uv[:,0]>=0)&(uv[:,0]<w)&(uv[:,1]>=0)&(uv[:,1]<h)
    return uv, valid

def sample_line(points, n=NUM_POINTS):
    points = np.asarray(points, dtype=np.float32)
    if points.ndim!=2 or points.shape[1]!=3 or len(points)<2 or not np.isfinite(points).all():
        raise ValueError('Invalid official 3D lane geometry.')
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(points,axis=0),axis=1))]
    if s[-1]<1e-7: raise ValueError('Degenerate lane.')
    s, keep = np.unique(s, return_index=True); points = points[keep]
    q = np.linspace(0,s[-1],n)
    return np.stack([np.interp(q,s,points[:,k]) for k in range(3)],-1).astype(np.float32)

manifest = json.loads(SPLIT_FILE.read_text())
records_by_split = {}
for split in ['train','val']:
    records = []
    for segment, timestamps in manifest[split].items():
        for timestamp in timestamps:
            timestamp = str(timestamp).split('.')[0]
            path = DATA_ROOT/split/str(segment)/'info'/f'{timestamp}.json'
            raw = json.loads(path.read_text())
            ann = raw['annotation']; lanes = ann['lane_centerline']
            if len(lanes)>MAX_LANES:
                raise ValueError(f'{path}: {len(lanes)} lanes exceed MAX_LANES; increase it, do not truncate.')
            adj = np.asarray(ann['topology_lclc'], dtype=np.float32).reshape(len(lanes),len(lanes))
            if not np.isin(adj,[0,1]).all(): raise ValueError(f'Invalid topology: {path}')
            camera = raw['sensor']['ring_front_center']
            image = DATA_ROOT/camera['image_path']
            if not image.is_file(): raise FileNotFoundError(image)
            records.append(dict(key=(split,str(segment),str(timestamp)), image=str(image),
                camera=camera, pose=rigid_transform(raw['pose']),
                points=[sample_line(l['points']) for l in lanes],
                ids=[l['id'] for l in lanes], adj=adj))
    records_by_split[split] = records
train_scenes = {r['key'][1] for r in records_by_split['train']}
val_scenes = {r['key'][1] for r in records_by_split['val']}
assert not train_scenes & val_scenes, 'Official train/val scenes overlap.'
print({k:len(v) for k,v in records_by_split.items()})


## 4. Causal sequences and temporal event masks
Each target frame is included, with earlier frames only. Scene starts repeat the first image with temporal supervision masked. Lane matching uses ego-motion compensated 3D geometry and Hungarian assignment. Ambiguous matches, unmatched lanes, merge/split neighborhoods, and changed topology are excluded from temporal consistency.

In [ ]:
def temporal_pairs(previous, current):
    a = np.asarray(previous['points'], dtype=np.float32)
    b = np.asarray(current['points'], dtype=np.float32)
    if len(a)==0 or len(b)==0: return np.empty((0,2),dtype=np.int64)
    # Previous ego -> world -> current ego.
    T = np.linalg.inv(current['pose']) @ previous['pose']
    a = a @ T[:3,:3].T + T[:3,3]
    cost = np.linalg.norm(a[:,None]-b[None],axis=-1).mean(-1)
    near = cost < TEMPORAL_GATE_METERS
    rows,cols = linear_sum_assignment(cost)
    matches = [(i,j) for i,j in zip(rows,cols) if near[i,j]
               and near[i].sum()==1 and near[:,j].sum()==1]
    if not matches: return np.empty((0,2),dtype=np.int64)
    ii,jj = np.array(matches).T
    A,B = previous['adj'],current['adj']
    result=[]
    for i,j in matches:
        branching = max(A[i].sum(),A[:,i].sum(),B[j].sum(),B[:,j].sum())>1
        changed = (not np.array_equal(A[i,ii],B[j,jj]) or
                   not np.array_equal(A[ii,i],B[jj,j]))
        # Degree changes also expose edges connected to unmatched lanes.
        changed |= A[i].sum()!=B[j].sum() or A[:,i].sum()!=B[:,j].sum()
        if not branching and not changed: result.append((i,j))
    return np.asarray(result,dtype=np.int64).reshape(-1,2)

class LaneSeqDS(Dataset):
    def __init__(self, records):
        self.windows=[]; groups=defaultdict(list)
        for r in records: groups[r['key'][1]].append(r)
        for scene,rs in sorted(groups.items()):
            rs.sort(key=lambda r:int(r['key'][2]))
            for end in range(len(rs)):
                indices=[max(0,k) for k in range(end-SEQ_LEN+1,end+1)]
                self.windows.append(([rs[k] for k in indices],indices))
        self.tf=transforms.Compose([transforms.Resize((IMAGE_SIZE[1],IMAGE_SIZE[0])),
            transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    def __len__(self): return len(self.windows)
    def __getitem__(self,index):
        rs,indices=self.windows[index]; ims=[]
        points=np.zeros((SEQ_LEN,MAX_LANES,NUM_POINTS,3),np.float32)
        mask=np.zeros((SEQ_LEN,MAX_LANES),bool)
        adj=np.zeros((SEQ_LEN,MAX_LANES,MAX_LANES),np.float32)
        pairs=np.full((SEQ_LEN-1,MAX_LANES,2),-1,np.int64)
        for t,r in enumerate(rs):
            with Image.open(r['image']) as im: ims.append(self.tf(im.convert('RGB')))
            count=len(r['points']); mask[t,:count]=True
            if count: points[t,:count]=(np.stack(r['points'])-XYZ_CENTER)/XYZ_SCALE
            adj[t,:count,:count]=r['adj']
            if t and indices[t]!=indices[t-1]:
                match=temporal_pairs(rs[t-1],r); pairs[t-1,:len(match)]=match
        return dict(images=torch.stack(ims),points=torch.from_numpy(points),
            mask=torch.from_numpy(mask),adj=torch.from_numpy(adj),pairs=torch.from_numpy(pairs),index=index)

train_ds=LaneSeqDS(records_by_split['train']); val_ds=LaneSeqDS(records_by_split['val'])
assert len(train_ds) and len(val_ds)
def seed_worker(worker_id):
    seed=torch.initial_seed()%2**32; random.seed(seed); np.random.seed(seed)
generator=torch.Generator().manual_seed(SEED)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,generator=generator,pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)


## 5. ConvNeXt V2 and Mamba-2 model
Learned queries attend to image spatial features. No labels, lane counts, or target points enter the forward pass. Mamba processes each query causally across frames; supervised temporal matching is applied separately in the loss.

In [ ]:
class LaneGraphNet(nn.Module):
    def __init__(self,d=256,pretrained=True):
        super().__init__()
        self.backbone=timm.create_model('convnextv2_tiny.fcmae_ft_in22k_in1k',
            pretrained=pretrained,features_only=True,out_indices=(3,))
        self.proj=nn.Conv2d(self.backbone.feature_info.channels()[-1],d,1)
        self.position=nn.Sequential(nn.Linear(2,d),nn.GELU(),nn.Linear(d,d))
        self.queries=nn.Parameter(torch.randn(MAX_LANES,d)*.02)
        self.cross=nn.MultiheadAttention(d,8,batch_first=True)
        self.graph=nn.MultiheadAttention(d,8,batch_first=True)
        self.norm=nn.LayerNorm(d)
        self.temporal=Mamba2(d_model=d,d_state=64,d_conv=4,expand=2,headdim=64)
        self.geom=nn.Linear(d,NUM_POINTS*3)
        self.logvar=nn.Linear(d,NUM_POINTS*3)
        self.presence=nn.Linear(d,1)
        self.edge=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Linear(d,2))
    def forward(self,images):
        B,T,C,H,W=images.shape
        f=self.proj(self.backbone(images.reshape(B*T,C,H,W))[-1])
        h,w=f.shape[-2:]
        yy,xx=torch.meshgrid(torch.linspace(-1,1,h,device=f.device),
            torch.linspace(-1,1,w,device=f.device),indexing='ij')
        position=self.position(torch.stack([xx,yy],-1).reshape(-1,2).to(f.dtype))
        memory=f.flatten(2).transpose(1,2)+position
        q=self.queries[None].expand(B*T,-1,-1)
        x=self.norm(q+self.cross(q,memory,memory,need_weights=False)[0])
        x=self.norm(x+self.graph(x,x,x,need_weights=False)[0])
        x=x.reshape(B,T,MAX_LANES,-1).permute(0,2,1,3).reshape(B*MAX_LANES,T,-1)
        x=self.norm(x+self.temporal(x))
        x=x.reshape(B,MAX_LANES,T,-1).permute(0,2,1,3)
        xi=x.unsqueeze(3).expand(-1,-1,-1,MAX_LANES,-1)
        xj=x.unsqueeze(2).expand(-1,-1,MAX_LANES,-1,-1)
        alpha=F.softplus(self.edge(torch.cat([xi,xj],-1)).float())+1
        return dict(geom=self.geom(x).reshape(B,T,MAX_LANES,NUM_POINTS,3),
            logvar=self.logvar(x).float().reshape(B,T,MAX_LANES,NUM_POINTS,3).clamp(-8,4),
            presence=self.presence(x).squeeze(-1),alpha=alpha,features=x)
model=LaneGraphNet().to(DEVICE)
print('Parameters:',sum(p.numel() for p in model.parameters()))


## 6. Assignment, evidential loss, and temporal consistency
Geometry uses diagonal Gaussian NLL in normalized ego coordinates. Connectivity uses a two-class Dirichlet (Beta) evidence head with expected classification error, variance regularization, and KL-to-uniform regularization. Geometry uncertainty is Gaussian, not Dirichlet.

In [ ]:
@torch.no_grad()
def assignments(out,batch):
    result={}
    for b in range(out['geom'].shape[0]):
        for t in range(out['geom'].shape[1]):
            ids=torch.where(batch['mask'][b,t])[0]
            if len(ids)==0: result[b,t]=(np.array([],int),np.array([],int)); continue
            pred=out['geom'][b,t].detach().float()
            gt=batch['points'][b,t,ids].to(pred.device)
            cost=(pred[:,None]-gt[None]).abs().mean((-1,-2))
            cost-=.1*out['presence'][b,t].sigmoid()[:,None]
            q,g=linear_sum_assignment(cost.cpu().numpy())
            result[b,t]=(q,ids[g].cpu().numpy())
    return result

def dirichlet_kl(alpha):
    total=alpha.sum(-1)
    return (torch.lgamma(total)-torch.lgamma(alpha).sum(-1)
        +((alpha-1)*(torch.digamma(alpha)-torch.digamma(total[...,None]))).sum(-1))

def losses(out,batch,anneal=1.):
    matched=assignments(out,batch); zero=out['geom'].sum()*0
    geo=[]; topo=[]; corr=[]
    target=torch.zeros_like(out['presence'])
    for (b,t),(q,g) in matched.items():
        if not len(q): continue
        target[b,t,q]=1
        gt=batch['points'][b,t,g].to(DEVICE)
        mu=out['geom'][b,t,q].float(); lv=out['logvar'][b,t,q]
        geo.append((.5*(lv+(mu-gt).square()*torch.exp(-lv))).mean())
        a=out['alpha'][b,t][q][:,q]
        y=batch['adj'][b,t][g][:,g].to(DEVICE).long()
        one=F.one_hot(y,2).float(); S=a.sum(-1,keepdim=True); prob=a/S
        error=((one-prob).square()+prob*(1-prob)/(S+1)).sum(-1)
        adjusted=one+(1-one)*a
        topo.append((error+.01*anneal*dirichlet_kl(adjusted)).mean())
    for b in range(out['geom'].shape[0]):
        for t in range(1,out['geom'].shape[1]):
            qp,gp=matched[b,t-1]; qc,gc=matched[b,t]
            previous=dict(zip(gp,qp)); current=dict(zip(gc,qc))
            for i,j in batch['pairs'][b,t-1].tolist():
                if i in previous and j in current:
                    f=out['features'][b,t-1,previous[i]].float()
                    h=out['features'][b,t,current[j]].float()
                    corr.append(1-F.cosine_similarity(f,h,dim=0))
    mean=lambda values: torch.stack(values).mean() if values else zero
    geometry,topology,temporal=mean(geo),mean(topo),mean(corr)
    presence=F.binary_cross_entropy_with_logits(out['presence'].float(),target)
    total=geometry+topology+presence+.1*temporal
    return total,dict(geometry=geometry.detach(),topology=topology.detach(),
        presence=presence.detach(),temporal=temporal.detach())


## 7. Training and measured runtime

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and AMP_DTYPE == torch.float16)
def epoch(loader,training,ep):
    model.train(training); sums=defaultdict(float); count=0
    for batch in loader:
        images=batch['images'].to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast('cuda',dtype=AMP_DTYPE,enabled=USE_AMP):
                out=model(images)
            loss,parts=losses(out,batch,min(1.,(ep+1)/10))
            if not torch.isfinite(loss): raise FloatingPointError('Nonfinite loss')
            if training:
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                scaler.step(optimizer); scaler.update()
        size=len(images); count+=size
        sums['loss']+=loss.item()*size
        for k,v in parts.items(): sums[k]+=v.item()*size
    return {k:v/count for k,v in sums.items()}

history=[]; best=float('inf'); torch.cuda.synchronize(); started=time.perf_counter()
for ep in range(EPOCHS):
    tr=epoch(train_loader,True,ep); va=epoch(val_loader,False,ep)
    torch.cuda.synchronize()
    row=dict(epoch=ep+1,elapsed_seconds=time.perf_counter()-started,
        **{'train_'+k:v for k,v in tr.items()},**{'val_'+k:v for k,v in va.items()})
    history.append(row); print(row)
    pd.DataFrame(history).to_csv(RUN_DIR/'training_history.csv',index=False)
    if va['loss']<best:
        best=va['loss']; torch.save(dict(model=model.state_dict(),optimizer=optimizer.state_dict(),
            scaler=scaler.state_dict(),epoch=ep+1,seed=SEED),RUN_DIR/'best.pt')
write_protocol(status='trained',training_seconds=time.perf_counter()-started,best_validation_loss=best)
checkpoint=torch.load(RUN_DIR/'best.pt',map_location=DEVICE,weights_only=False)
model.load_state_dict(checkpoint['model']); model.eval()


## 8. Uncertainty, calibration, and coverage
Calibration diagnostics use Hungarian-matched validation lanes in each window’s final frame. They are conditional diagnostics, not official benchmark metrics. Presence errors remain part of official detection evaluation. Thresholds are fixed in configuration, never tuned on the test set.

In [ ]:
@torch.no_grad()
def infer(images):
    model.eval()
    with torch.autocast('cuda',dtype=AMP_DTYPE,enabled=USE_AMP): out=model(images.to(DEVICE))
    a=out['alpha']; prob=a[...,1]/a.sum(-1)
    out.update(probability=prob,vacuity=2/a.sum(-1),
        entropy=-(prob.clamp_min(1e-8)*prob.clamp_min(1e-8).log()
            +(1-prob).clamp_min(1e-8)*(1-prob).clamp_min(1e-8).log()),
        lane_probability=out['presence'].sigmoid())
    return out

def expected_calibration_error(p,y,bins=15):
    if not len(p): return None
    ids=np.minimum((p*bins).astype(int),bins-1)
    return float(sum((ids==k).mean()*abs(p[ids==k].mean()-y[ids==k].mean())
        for k in range(bins) if (ids==k).any()))

@torch.no_grad()
def calibration(loader):
    probabilities=[]; labels=[]; covered=[]; widths=[]
    for batch in loader:
        out=infer(batch['images']); match=assignments(out,batch)
        for b in range(len(batch['images'])):
            q,g=match[b,SEQ_LEN-1]
            if not len(q): continue
            p=out['probability'][b,-1][q][:,q].cpu().numpy()
            y=batch['adj'][b,-1][g][:,g].numpy()
            probabilities.extend(p.ravel()); labels.extend(y.ravel())
            mu=out['geom'][b,-1,q].float().cpu().numpy()
            std=(.5*out['logvar'][b,-1,q]).exp().cpu().numpy()
            gt=batch['points'][b,-1,g].numpy()
            covered.append(np.abs(mu-gt)<=1.96*std)
            widths.append(2*1.96*std*XYZ_SCALE)
    p=np.asarray(probabilities); y=np.asarray(labels)
    result=dict(edge_ece=expected_calibration_error(p,y),
        edge_brier=float(np.mean((p-y)**2)) if len(p) else None,
        geometry_marginal_95_coverage_xyz=np.concatenate(covered).mean((0,1)).tolist() if covered else None,
        geometry_mean_95_interval_width_m_xyz=np.concatenate(widths).mean((0,1)).tolist() if widths else None)
    (RUN_DIR/'calibration.json').write_text(json.dumps(result,indent=2)); return result
print(calibration(val_loader))


## 9. Official evaluator export
Exports final-frame predictions for every official validation frame, in original ego-frame metres with consistent topology ordering. Empty traffic predictions explicitly represent the absent branch. The official evaluator determines the available lane metrics; any aggregate OLS from this lane-only model includes missing traffic performance and must not be described as a full-system result. Use the matching devkit release and trusted collection pickle.

In [ ]:
@torch.no_grad()
def export_predictions():
    results={}
    for batch in val_loader:
        out=infer(batch['images'])
        for b,index in enumerate(batch['index'].tolist()):
            record=val_ds.windows[index][0][-1]
            selected=torch.where(out['lane_probability'][b,-1]>=LANE_THRESHOLD)[0]
            xyz=out['geom'][b,-1,selected].float().cpu().numpy()*XYZ_SCALE+XYZ_CENTER
            scores=out['lane_probability'][b,-1,selected].float().cpu().numpy()
            lanes=[dict(id=i,points=p.astype(np.float32),confidence=float(s))
                for i,(p,s) in enumerate(zip(xyz,scores))]
            topology=out['probability'][b,-1][selected][:,selected].float().cpu().numpy()
            results[record['key']]=dict(predictions=dict(lane_centerline=lanes,
                traffic_element=[],topology_lclc=topology,
                topology_lcte=np.zeros((len(lanes),0),np.float32)))
    assert set(results)=={r['key'] for r in records_by_split['val']}
    # Optional leaderboard author/contact fields must be filled by the researcher.
    return dict(method='ConvNeXtV2-Mamba2-LaneGraph',authors=[],results=results)

submission=export_predictions()
import pickle
with (RUN_DIR/'validation_predictions.pkl').open('wb') as file:
    pickle.dump(submission,file)
# Official API (centerline task): matching and scoring are owned by the devkit.
from openlanev2.centerline.evaluation import evaluate
assert VAL_COLLECTION.is_file(), 'Set VAL_COLLECTION to the official validation collection.'
with VAL_COLLECTION.open('rb') as file: ground_truth=pickle.load(file)
assert set(ground_truth)==set(submission['results']), 'Collection does not match selected official split.'
metrics=evaluate(ground_truth,submission)
print(metrics)
(RUN_DIR/'official_metrics.json').write_text(json.dumps(metrics,indent=2,
    default=lambda x: x.tolist() if isinstance(x,np.ndarray) else float(x)))


## 10. Project predictions for visual inspection
Projection is for visualization only. Training and benchmark export retain 3D coordinates.

Implementation references: [official schema and splits](https://github.com/OpenDriveLab/OpenLane-V2/blob/master/data/README.md), [OpenLane-V2 devkit](https://github.com/OpenDriveLab/OpenLane-V2), [Mamba-2](https://github.com/state-spaces/mamba), [timm ConvNeXt V2](https://huggingface.co/timm/convnextv2_tiny.fcmae_ft_in22k_in1k).

In [ ]:
import matplotlib.pyplot as plt
@torch.no_grad()
def visualize(index=0):
    item=val_ds[index]; record=val_ds.windows[index][0][-1]
    out=infer(item['images'][None])
    with Image.open(record['image']) as im:
        image=np.asarray(im.convert('RGB')); h,w=image.shape[:2]
    fig,ax=plt.subplots(figsize=(12,6)); ax.imshow(image)
    def draw(points,color,style):
        uv,valid=project_ego(points,record['camera'],(w,h))
        uv[~valid]=np.nan; ax.plot(uv[:,0],uv[:,1],color+style,lw=1.2)
    for points in record['points']: draw(points,'g','--')
    for q in torch.where(out['lane_probability'][0,-1]>=LANE_THRESHOLD)[0]:
        points=out['geom'][0,-1,q].float().cpu().numpy()*XYZ_SCALE+XYZ_CENTER
        draw(points,'r','-')
    ax.set_title('Green: ground truth; red: image-only predictions'); ax.axis('off'); plt.show()
visualize()
